# Master Pareto plot

Phase 4 deliverable: the canonical compute–accuracy figure of the paper.
Aggregates every per-config evaluation into a single overlay so we can
tell a clean story about where the cascade frontier sits relative to
monolithic detectors.

Inputs (all produced by `scripts/eval_cascade.py`):

- `reports/cascade/<gate>_<calibration>.json` for every gate × calibration combination
- `reports/cascade/oracle.json` — perfect gate baseline (upper bound at gate cost = 0)
- `reports/cascade/yolo11m_baseline.json` — monolithic detector at varying input sizes
- `reports/cascade/codesign_*.json` — Pillar 3 co-design variants

Output: `paper_assets/figures/master_pareto.{png,pdf}` and a CSV companion.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

REPORTS = Path('../reports/cascade')
FIG_DIR = Path('../paper_assets/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path('../paper_assets/tables')
TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
rows = []
for path in sorted(REPORTS.glob('*.json')):
    for r in json.loads(path.read_text()):
        r['source_file'] = path.stem
        rows.append(r)
df = pd.DataFrame(rows)
if 'mAP@0.50' not in df.columns and 'mAP@0.5' in df.columns:
    df = df.rename(columns={'mAP@0.5': 'mAP@0.50'})
df['series'] = df['label'].fillna('cascade') + '|' + df['calibration'].fillna('identity')
df.to_csv(TABLES_DIR / 'master_pareto.csv', index=False)
print(f'{len(df)} rows from {df.source_file.nunique()} files')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for series, group in df.groupby('series'):
    g = group.sort_values('total_gflops')
    style = '--' if 'oracle' in series else '-'
    marker = 'o'
    ax.plot(g['total_gflops'], g['mAP@0.50'], linestyle=style, marker=marker, label=series, linewidth=1.6)
ax.set_xscale('log')
ax.set_xlabel('Total GFLOPs per image (log)')
ax.set_ylabel('mAP@0.5 (DOTA-Ships, geographic split)')
ax.set_title('Cascade compute–accuracy frontier')
ax.grid(True, alpha=0.3, which='both')
ax.legend(fontsize=7, loc='lower right')
fig.tight_layout()
fig.savefig(FIG_DIR / 'master_pareto.png', dpi=200)
fig.savefig(FIG_DIR / 'master_pareto.pdf')
plt.show()

## Stratified Pareto (supplementary)

If `eval_cascade.py` was run with `--stratum {gsd_bucket|size_bucket|imagesource}`,
load the `*.stratified_<stratum>.json` files and plot one panel per bucket.

In [ ]:
from itertools import groupby
stratified = []
for path in sorted(REPORTS.glob('*.stratified_*.json')):
    for r in json.loads(path.read_text()):
        r['source_file'] = path.stem
        stratified.append(r)
if stratified:
    sdf = pd.DataFrame(stratified)
    print(f'{len(sdf)} stratified rows; buckets = {sorted(sdf.bucket.unique())}')
    sdf.to_csv(TABLES_DIR / 'stratified_pareto.csv', index=False)
    sdf.head()
else:
    print('No stratified eval files yet. Run eval_cascade.py with --stratum.')

## Cross-dataset robustness (HRSC2016)

Apply the best frozen cascade configuration zero-shot to HRSC2016 and report
the degradation versus the in-distribution DOTA-Ships number.

In [ ]:
hrsc_path = REPORTS / 'hrsc2016_zero_shot.json'
if hrsc_path.exists():
    hrsc = pd.DataFrame(json.loads(hrsc_path.read_text()))
    hrsc.to_csv(TABLES_DIR / 'hrsc_zero_shot.csv', index=False)
    print(hrsc[['threshold', 'mAP@0.50', 'filter_rate', 'gate_recall_on_positive_tiles']])
else:
    print('Run scripts/eval_cascade.py against HRSC2016 prepared data and re-run this cell.')